In [16]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split #type:ignore
from sklearn.preprocessing import StandardScaler #type:ignore
import os
import sys
from google.colab import drive #type:ignore
import random

In [ ]:
try:
    drive.mount('/content/drive')
    Colab = True
except:
    Colab = False
if Colab:
    data_dir='/content/drive/MyDrive/AI/SpotifyRecommender/Data'
else:
    data_dir='./Data'
    
if Colab:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

In [17]:
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

In [ ]:
data=pd.read_csv(os.path.join(data_dir,'dataset.csv'))
data.shape

In [ ]:
# Prepare categorical and numeric features
from sklearn.preprocessing import LabelEncoder

# Encode categorical features
artist_encoder = LabelEncoder()
genre_encoder = LabelEncoder()

data['artist_encoded'] = artist_encoder.fit_transform(data['artists'])
data['genre_encoded'] = genre_encoder.fit_transform(data['track_genre'])

# Numeric audio features
numeric_cols = ['danceability', 'energy', 'key', 'loudness', 'mode', 
                'speechiness', 'acousticness', 'instrumentalness', 
                'liveness', 'valence', 'tempo', 'duration_ms', 'popularity']

# Scale numeric features
scaler = StandardScaler()
data_numeric = scaler.fit_transform(data[numeric_cols])
data_numeric = torch.tensor(data_numeric, dtype=torch.float32)

# Categorical features as tensors
data_artists = torch.tensor(data['artist_encoded'].values, dtype=torch.long)
data_genres = torch.tensor(data['genre_encoded'].values, dtype=torch.long)

num_artists = len(artist_encoder.classes_)
num_genres = len(genre_encoder.classes_)

print(f"Numeric features shape: {data_numeric.shape}")
print(f"Number of unique artists: {num_artists}")
print(f"Number of unique genres: {num_genres}")

In [ ]:
# Dataset for training pairs
class SongPairDataset(Dataset):
    def __init__(self, numeric_features, artist_ids, genre_ids, data, num_samples=50000):
        self.numeric_features = numeric_features
        self.artist_ids = artist_ids
        self.genre_ids = genre_ids
        self.data = data
        self.pairs = []
        self.labels = []
        
        # Generate positive pairs (same genre)
        for _ in range(num_samples // 2):
            genre = data['genre_encoded'].sample(1).values[0]
            same_genre = data[data['genre_encoded'] == genre]
            if len(same_genre) >= 2:
                idx1, idx2 = np.random.choice(same_genre.index, 2, replace=False)
                self.pairs.append((idx1, idx2))
                self.labels.append(1.0)
        
        # Generate negative pairs (different genres)
        for _ in range(num_samples // 2):
            idx1, idx2 = np.random.choice(len(data), 2, replace=False)
            if data.iloc[idx1]['genre_encoded'] != data.iloc[idx2]['genre_encoded']:
                self.pairs.append((idx1, idx2))
                self.labels.append(0.0)
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        idx1, idx2 = self.pairs[idx]
        return (self.numeric_features[idx1], self.artist_ids[idx1], self.genre_ids[idx1],
                self.numeric_features[idx2], self.artist_ids[idx2], self.genre_ids[idx2],
                torch.tensor(self.labels[idx], dtype=torch.float32))

print("Dataset class defined!")

In [ ]:
# Transformer-based encoder with categorical embeddings
class HybridSongEncoder(nn.Module):
    def __init__(self, num_artists, num_genres, num_numeric=13, artist_emb_dim=32, genre_emb_dim=16, d_model=128, nhead=4, num_layers=2, embedding_dim=64):
        super(HybridSongEncoder, self).__init__()
        
        # Embedding layers for categorical features
        self.artist_embedding = nn.Embedding(num_artists, artist_emb_dim)
        self.genre_embedding = nn.Embedding(num_genres, genre_emb_dim)
        
        # Total input dimension after concatenation
        total_input_dim = num_numeric + artist_emb_dim + genre_emb_dim
        
        # Project to d_model
        self.input_projection = nn.Linear(total_input_dim, d_model)
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=256,
            batch_first=True,
            dropout=0.1
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Output projection
        self.output_projection = nn.Sequential(
            nn.Linear(d_model, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, embedding_dim)
        )
        
    def forward(self, numeric, artist_ids, genre_ids):
        # Get embeddings for categorical features
        artist_emb = self.artist_embedding(artist_ids)  # (batch, artist_emb_dim)
        genre_emb = self.genre_embedding(genre_ids)      # (batch, genre_emb_dim)
        
        # Concatenate all features
        x = torch.cat([numeric, artist_emb, genre_emb], dim=1)  # (batch, total_input_dim)
        
        # Add sequence dimension for transformer
        x = x.unsqueeze(1)  # (batch, 1, total_input_dim)
        
        # Project and pass through transformer
        x = self.input_projection(x)  # (batch, 1, d_model)
        x = self.transformer(x)       # (batch, 1, d_model)
        
        # Remove sequence dimension and project to embedding
        x = x.squeeze(1)  # (batch, d_model)
        embedding = self.output_projection(x)  # (batch, embedding_dim)
        
        # Normalize embeddings for cosine similarity
        embedding = F.normalize(embedding, p=2, dim=1)
        
        return embedding


# Siamese network
class SiameseRecommender(nn.Module):
    def __init__(self, num_artists, num_genres, num_numeric=13, embedding_dim=64):
        super(SiameseRecommender, self).__init__()
        self.encoder = HybridSongEncoder(num_artists, num_genres, num_numeric, embedding_dim=embedding_dim)
    
    def forward(self, num1, art1, gen1, num2, art2, gen2):
        # Encode both songs
        emb1 = self.encoder(num1, art1, gen1)
        emb2 = self.encoder(num2, art2, gen2)
        
        # Compute cosine similarity
        similarity = F.cosine_similarity(emb1, emb2, dim=1)
        
        return similarity
    
    def encode(self, numeric, artist_ids, genre_ids):
        return self.encoder(numeric, artist_ids, genre_ids)

print("Hybrid model with embeddings + transformer defined!")

In [ ]:
# Create dataset and train
print("Creating training dataset...")
train_dataset = SongPairDataset(data_numeric, data_artists, data_genres, data, num_samples=50000)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

print(f"Training samples: {len(train_dataset)}")

# Initialize model
model = SiameseRecommender(num_artists=num_artists, num_genres=num_genres, 
                          num_numeric=len(numeric_cols), embedding_dim=64)
model = model.to(device)

# Training setup
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCEWithLogitsLoss()

print("\nStarting training...")
model.train()
num_epochs = 5

for epoch in range(num_epochs):
    total_loss = 0
    for batch_idx, (num1, art1, gen1, num2, art2, gen2, labels) in enumerate(train_loader):
        num1, art1, gen1 = num1.to(device), art1.to(device), gen1.to(device)
        num2, art2, gen2 = num2.to(device), art2.to(device), gen2.to(device)
        labels = labels.to(device)
        
        # Forward pass
        similarity = model(num1, art1, gen1, num2, art2, gen2)
        loss = criterion(similarity, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        if batch_idx % 50 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Batch [{batch_idx}/{len(train_loader)}], Loss: {loss.item():.4f}")
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] Average Loss: {avg_loss:.4f}\n")

print("Training completed!")

In [ ]:
# Encode all songs
print("Encoding all songs...")
model.eval()

with torch.no_grad():
    all_embeddings = []
    batch_size = 1024
    
    for i in range(0, len(data_numeric), batch_size):
        end_idx = min(i + batch_size, len(data_numeric))
        num_batch = data_numeric[i:end_idx].to(device)
        art_batch = data_artists[i:end_idx].to(device)
        gen_batch = data_genres[i:end_idx].to(device)
        
        embeddings = model.encode(num_batch, art_batch, gen_batch)
        all_embeddings.append(embeddings.cpu())
    
    all_embeddings = torch.cat(all_embeddings, dim=0)

print(f"Embeddings shape: {all_embeddings.shape}")
print("Ready for recommendations!")

In [ ]:
# Recommendation function using learned embeddings
def get_recommendations(song_name, top_n=10):
    # Find the song
    matches = data[data['track_name'] == song_name]
    
    if len(matches) == 0:
        return f"Song '{song_name}' not found in dataset"
    
    idx = matches.index[0]
    
    # Get the embedding for this song
    query_embedding = all_embeddings[idx].unsqueeze(0)
    
    # Compute cosine similarity with all songs
    similarities = F.cosine_similarity(query_embedding, all_embeddings, dim=1)
    
    # Exclude the query song itself
    similarities[idx] = -1
    
    # Get top N
    top_indices = torch.topk(similarities, k=top_n).indices.numpy()
    top_scores = torch.topk(similarities, k=top_n).values.numpy()
    
    # Create recommendations dataframe
    recommendations = data.iloc[top_indices][['track_name', 'artists', 'track_genre']].copy()
    recommendations['similarity_score'] = top_scores
    recommendations['id'] = top_indices    
    return recommendations

print("Recommendation function ready!")

In [ ]:
# Test the recommender
song_id=1
test_song = data['track_name'].iloc[song_id]
print(f"Getting recommendations for: {test_song}")
print(f"Artist: {data['artists'].iloc[song_id]}")
print(f"Genre: {data['track_genre'].iloc[song_id]}")
print("\n" + "="*80)
print("Top 10 Recommendations (Neural Network with Transformer + Embeddings):")
print("="*80 + "\n")

recommendations = get_recommendations(test_song, top_n=3) 
print(recommendations.to_string(index=False))

In [27]:
# TODO This is not workin atm
song_queue=[]
liked_song=[]
next_song=data['track_name'].iloc[random.randint(0, len(data)-1)]
while True:
    liked=bool(int(input("\nDid you like the recommendations? (1 for Yes, 0 for No): ")))
    next_id=None
    if liked:
        liked_song.append(next_song)
        song_queue.extend(recommendations['track_name'].tolist())
        next_song=song_queue.pop(0)
    else:
        if len(song_queue)>0:
            next_song=song_queue.pop(0)
        else:
            print("No more recommendations in the queue.")
            break
    recommendations = get_recommendations(next_song,top_n=3)
    print(f'\rliked songs: {liked_song}\nsong queue: {song_queue}\nnext song: {next_song}\r')    

liked songs: ["I'll Take You There - Moplen Radio Version"]
song queue: ['Everytime We Touch (Acoustic)', '恋の魔力']
next song: My Heart
liked songs: ["I'll Take You There - Moplen Radio Version"]
song queue: ['恋の魔力']
next song: Everytime We Touch (Acoustic)
liked songs: ["I'll Take You There - Moplen Radio Version"]
song queue: []
next song: 恋の魔力
No more recommendations in the queue.
